# Resolve a storage code explicitly

The database column is called money_unit; the domain field is called currency.

## Prepare the model

Run this cell first. It imports the required types and defines the model used by this example. Start with a fresh repository Python kernel so declarations from other examples do not enter this model.

In [1]:
from pydantic import Field

from aoa.action_machine.domain import BaseCoreEntity, BaseEnumEntity, value
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


class ShopDomain(BaseDomain):
    """Group the shop declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Currencies accepted by the shop", domain=ShopDomain)
class CurrencyEnumEntity(BaseEnumEntity):
    """Declare the shop's currency vocabulary."""

    symbol: str = Field(description="Currency symbol")

    EUR = value(code="eur", display_name="Euro", symbol="€")
    USD = value(code="usd", display_name="US dollar", symbol="$")


@entity(description="Price tag", domain=ShopDomain)
class PriceTagCoreEntity(BaseCoreEntity):
    """Associate an amount with its currency."""

    id: str = Field(description="Tag identifier")
    amount: float = Field(description="Price amount")
    currency: CurrencyEnumEntity = Field(description="Money unit")

## Run the example

The following cell performs the operation described above. Its assertions check the result; any expected refusal is caught explicitly.

In [2]:
from aoa.action_machine.domain import build

row = {"tag_key": "tag-1", "price": 9.99, "money_unit": "eur"}
selected = CurrencyEnumEntity.by_code(row["money_unit"])
tag = build(
    row,
    PriceTagCoreEntity,
    lambda fields, data: {
        fields.id: data["tag_key"],
        fields.amount: data["price"],
        fields.currency: selected,
    },
)
print(tag.id, tag.currency.code)
print("Shared value:", tag.currency is CurrencyEnumEntity.EUR)

tag-1 eur
Shared value: True


## What the result means

The resource translates storage column names into domain fields. It resolves `money_unit` through `by_code()` before passing the selected member to `build()`. The price keeps the shared declared object; the mapping does not add a new currency.